# 🧪 Pipeline Complet : Feature Engineering & Benchmark de Modèles

Ce notebook s'inscrit dans la continuité de l'analyse exploratoire (**01_eda.ipynb**). 
Maintenant que nous avons compris la structure des données, nous allons :
1.  **Recharger les données** via notre pipeline scripté (`src/preprocessing.py`) qui implémente le nettoyage et la création de variables identifiés lors de l'EDA.
2.  **Tester systématiquement** une large gamme de modèles pour identifier le plus performant :
    *   **Modèles Linéaires** : Linear Regression, Ridge, Lasso, ElasticNet.
    *   **Modèles Non-Linéaires** : Arbres de décision.
    *   **Ensemble Methods** : Random Forest.
    *   **Boosting** : Gradient Boosting (Sklearn), XGBoost, et LightGBM.

In [ ]:
import sys
import os
import time
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

# Métriques et Modèles Sklearn
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Boosting Avancé
import xgboost as xgb
import lightgbm as lgb

# Ajout du chemin src pour importer preprocessing
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
import preprocessing

plt.style.use('ggplot')
sns.set_context("notebook", font_scale=1.2)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

## 1. 🛠️ Chargement et Préparation des Données

Nous utilisons la fonction `get_dynamic_datasets` de notre module `preprocessing`. 
Cette fonction :
*   Charge les CSV.
*   Filtre sur les semaines pertinentes (Last 10 weeks).
*   Effectue les jointures (Oil, Holidays, Transactions).
*   Génère les Features (Lags, Rolling Means, Date features, Target Encoding).

In [ ]:
base_path = os.path.join("..", "data")
print("🚀 Lancement du Pipeline de Prétraitement...")

# Chargement (Train = ~7 semaines, Valid = ~2 semaines)
train_df, valid_df = preprocessing.get_dynamic_datasets(base_path)

print(f"\nDimension Train : {train_df.shape}")
print(f"Dimension Valid : {valid_df.shape}")

In [ ]:
# Conversion en Pandas pour compatibilité Scikit-Learn
# On applique aussi une transformation Log(1+x) sur la cible pour utiliser la RMSE comme métrique (équivalent NWRMSLE)

def prepare_sklearn_data(df, target_col="unit_sales_win"):
    # Colonnes à exclure (non-features)
    drop_cols = ["date", "id", "unit_sales", "unit_sales_win"]
    
    # Sélection automatique des colonnes numériques
    features = [c for c in df.columns 
                if c not in drop_cols 
                and df[c].dtype in [pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.UInt8, pl.UInt32]]
    
    # Conversion
    X = df.select(features).to_pandas()
    # Remplacer les valeurs manquantes par 0 (sécurité pour modèles linéaires)
    X = X.fillna(0)
    
    # Cible Log-transformée
    y = np.log1p(df[target_col].to_numpy())
    
    return X, y, features

print("Conversion en formats Numpy/Pandas...")
X_train, y_train, feature_cols = prepare_sklearn_data(train_df)
X_valid, y_valid, _ = prepare_sklearn_data(valid_df)

print(f"Nb Features : {len(feature_cols)}")
print(f"Features : {feature_cols}")

## 2. 🏎️ Benchmark des Modèles

Nous allons définir un dictionnaire de modèles à tester, allant du plus simple au plus complexe.

In [ ]:
# Dictionnaire des modèles
models_dict = {
    # --- Modèles Linéaires ---
    "Linear Regression": LinearRegression(),
    "Ridge (L2)": Ridge(alpha=1.0), 
    "Lasso (L1)": Lasso(alpha=0.01), # Alpha faible pour ne pas tout tuer
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5),
    
    # --- Arbres & Forêts ---
    "Decision Tree": DecisionTreeRegressor(max_depth=15, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=30, max_depth=15, n_jobs=-1, random_state=42), 
    # Note: RF réduit (30 arbres) pour limiter le temps de calcul dans le notebook
    
    # --- Boosting ---
    "Gradient Boosting (Sklearn)": GradientBoostingRegressor(n_estimators=50, max_depth=7, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=100, max_depth=10, learning_rate=0.05, n_jobs=-1, random_state=42),
    "LightGBM": lgb.LGBMRegressor(n_estimators=1000, num_leaves=31, learning_rate=0.05, n_jobs=-1, random_state=42)
}

### Lancement de la boucle d'entraînement
Attention : Certains modèles (Random Forest, Gradient Boosting) peuvent être longs à entraîner sur de gros volumes.
Nous mesurons le temps d'entraînement pour évaluer le compromis Performance / Coût.

In [ ]:
results = []

print(f"{'Modèle':<30} | {'RMSE':<10} | {'Time (s)':<10}")
print("-"*60)

for name, model in models_dict.items():
    # Mesure du temps
    start_time = time.time()
    
    # Entraînement
    # Cas spécifique pour l'Early Stopping de LightGBM
    if name == "LightGBM":
         model.fit(X_train, y_train, 
                   eval_set=[(X_valid, y_valid)], 
                   eval_metric="rmse",
                   callbacks=[lgb.early_stopping(50, verbose=False)])

    elif name == "XGBoost":
        model.fit(X_train, y_train, 
                  eval_set=[(X_valid, y_valid)], 
                  verbose=False)
    else:
        model.fit(X_train, y_train)
    
    end_time = time.time()
    elapsed = end_time - start_time
    
    # Prédiction
    y_pred = model.predict(X_valid)
    y_pred = np.maximum(y_pred, 0) # Pas de log négatif impossible, mais clip à 0 par sécurité
    
    # Calculs métriques
    rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
    mae = mean_absolute_error(y_valid, y_pred)
    
    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "Time (s)": elapsed
    })
    
    print(f"{name:<30} | {rmse:.5f}    | {elapsed:.2f} s")

## 3. 📊 Analyse et Comparaison des Résultats

In [ ]:
# Création du DataFrame de résultats
df_res = pd.DataFrame(results).sort_values("RMSE")

# Visualisation RMSE
plt.figure(figsize=(14, 8))
sns.barplot(x="RMSE", y="Model", data=df_res, palette="viridis")
plt.title("Comparaison des Modèles (RMSE sur Log-Sales) - Plus bas est meilleur")
plt.xlabel("RMSE")
plt.show()

# Visualisation Temps de Calcul
plt.figure(figsize=(14, 8))
sns.barplot(x="Time (s)", y="Model", data=df_res, palette="magma")
plt.title("Temps d'Entraînement par Modèle (Secondes)")
plt.xlabel("Temps (s)")
plt.show()

df_res

### Conclusion de l'Exploration

1.  **Modèles Linéaires** : Ils offrent une baseline rapide mais peinent souvent à capturer les relations non-linéaires complexes des séries temporelles (saturations, effets de seuil).
2.  **Arbres et Forêts** : Meilleure performance grâce à la gestion des non-linéarités, mais souvent plus lents (Random Forest) et sujets à l'overfitting si non contraints.
3.  **Boosting (XGBoost / LightGBM)** : 
    *   Ils dominent généralement ce type de compétition Kaggle.
    *   **LightGBM** est particulièrement recommandé ici pour sa rapidité d'exécution sur de grands volumes de données et sa capacité à gérer nativement les catégories (via le target encoding que nous avons fait ou son support natif).

C'est donc logiquement le **LightGBM** que nous avons retenu pour le pipeline de production (`src/train.py`).